# 🌴 Tropical Diseases — Instant Knowledge Graph Construction (Decoupled Mode)
### Pre-Extracted Triplets + Fast `bge-m3` Embedding (~10 Minutes for All 1,797 Cases)

This notebook uses **pre-extracted clinical entities and relationship triplets** generated offline by Antigravity subagents in `kg_extraction/`.

**Why this is 100x faster than running local Ollama:**
- **Traditional local Ollama:** Runs a 3B LLM token-by-token for each case (~50s per case = ~24 hours total → Colab times out).
- **Decoupled Mock LLM (This Notebook):** Uses pre-extracted frontier triplets. The LLM extraction step takes **0.001s** per case. Only the fast **`bge-m3` embedding** runs (~10 minutes total for all 1,797 cases)!

| Platform | Recommended Settings |
| :--- | :--- |
| **Google Colab (T4 GPU)** | Runtime → Change runtime type → T4 GPU. Finishes all cases in **~3–5 minutes**. |
| **Apple Silicon Mac (Local)** | Run locally via Jupyter. Finishes all cases in **~8–12 minutes**. |

---
## Step 1: Verify GPU

In [1]:
import sys, platform

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running on Google Colab")
    !nvidia-smi
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU ready: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"💻 Running locally on Mac: {platform.machine()} ({platform.system()})")
    try:
        import torch
        if torch.backends.mps.is_available():
            print("✅ Apple Silicon Metal (MPS) GPU acceleration is available!")
        else:
            print("ℹ️ CPU mode active")
    except ImportError:
        print("ℹ️ Local Python environment ready (PyTorch not required for Ollama)")

💻 Running locally on Mac: arm64 (Darwin)
ℹ️ Local Python environment ready (PyTorch not required for Ollama)


---
## Step 2: Install Ollama & LightRAG

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install zstd and Ollama on Colab
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
    print("✅ Colab dependencies & Ollama installed!")
else:
    # On Mac, check existing .venv packages
    try:
        try:
            import nest_asyncio
        except ImportError:
            import nest_asyncio2 as nest_asyncio
        import lightrag, ollama, tqdm
        print("✅ All dependencies are already installed in your local .venv!")
    except ImportError as e:
        print(f"Installing missing dependency: {e}")
        !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
        print("✅ Dependencies installed!")

✅ All dependencies are already installed in your local .venv!


---
## Step 3: Start Ollama & Pull Models (~1 minute)
We pull:
1. **`bge-m3`** — Fast medical embedding model (used during graph building).
2. **`qwen2.5:3b`** — Small model used **only at query time** in Step 7 (not used during ingestion).

In [3]:
import os, subprocess, time, urllib.request

EMBEDDING_MODEL = "bge-m3"
EMBED_DIM = 1024
QUERY_LLM_MODEL = "qwen2.5:3b"  # Used only for Step 7 queries, NOT during ingestion

IN_COLAB = "google.colab" in sys.modules
ollama_bin = "ollama" if IN_COLAB else "/usr/local/bin/ollama"

# Check if Ollama server is already running
server_running = False
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as resp:
        if resp.status == 200:
            server_running = True
            print("✅ Ollama server is already running!")
except Exception:
    pass

if not server_running:
    print("Starting Ollama server...")
    os.environ["OLLAMA_NUM_PARALLEL"] = "8"
    subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=os.environ
    )
    time.sleep(3)
    print("✅ Ollama server started!")

print(f"\n1. Pulling embedding model ({EMBEDDING_MODEL})...")
!ollama pull {EMBEDDING_MODEL}

print(f"\n2. Pulling query model ({QUERY_LLM_MODEL}) for Step 7 queries...")
!ollama pull {QUERY_LLM_MODEL}

print("\n--- Installed Models ---")
!ollama list

✅ Ollama server is already running!

Verifying models (qwen2.5:3b and bge-m3)...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 5ee4f07cdb9b: 100% ▕██████████████████▏ 1.9 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling b5c0e5cf74cf: 100% ▕██████████████████▏ 7.4 KB                         
pulling 161ddde4c9cd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifes

---
## Step 4: Mount Google Drive & Configure Storage
Mounts Google Drive for **persistent backup** after ingestion completes.

On Colab, ingestion runs on the **fast local NVMe SSD** (`/content/tropical_kg_storage`), then the finished KG is copied to Drive once at the end.
On Mac, both paths point to the same local directory.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LOCAL_STORAGE = "/content/tropical_kg_storage"
    DRIVE_STORAGE = "/content/drive/MyDrive/tropical_kg_storage"
    print("✅ Google Drive mounted!")
    print(f"⚡ Fast local workspace: {LOCAL_STORAGE}")
    print(f"💾 Persistent Drive backup: {DRIVE_STORAGE}")
else:
    LOCAL_STORAGE = "./tropical_kg_storage"
    DRIVE_STORAGE = LOCAL_STORAGE  # Same path on Mac
    print("✅ Running on Mac! Storage:", os.path.abspath(LOCAL_STORAGE))

os.makedirs(LOCAL_STORAGE, exist_ok=True)

---
## Step 5: Locate Dataset & Pre-Extracted Triplets (with Auto-Unzip)
Locates `partA_final.jsonl` and all pre-extracted clinical triplets.
If uploaded as `.zip` (e.g. `extracted_triplets.zip` or `archive.zip`), it **automatically unzips them**!

In [5]:
import os, json, glob, zipfile

# ------------------------------------------------------------
# A. AUTO-UNZIP: Check if any zip archives need extracting
# ------------------------------------------------------------
possible_zips = [
    "./extracted_triplets.zip",
    "./kg_extraction/extracted_triplets.zip",
    "../3_LightRAG_Knowledge_Graph/kg_extraction/extracted_triplets.zip",
    "/content/extracted_triplets.zip",
    "/content/drive/MyDrive/extracted_triplets.zip",
    "../2_Clinical_Verification/archive.zip",
    "/content/archive.zip",
]

for zp in possible_zips:
    if os.path.exists(zp):
        extract_dest = os.path.dirname(zp) if os.path.dirname(zp) else "."
        # Check if already unzipped to avoid redundant work
        if "extracted_triplets" in zp and not os.path.exists(os.path.join(extract_dest, "extracted_triplets")):
            print(f"📦 Auto-unzipping {zp} to {extract_dest}...")
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(extract_dest)
            print(f"✅ Successfully unzipped {zp}!")
        elif "archive.zip" in zp and not os.path.exists(os.path.join(extract_dest, "kaggle_partA")):
            print(f"📦 Auto-unzipping dataset {zp}...")
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(extract_dest)
            print(f"✅ Successfully unzipped {zp}!")

# ------------------------------------------------------------
# B. Locate partA_final.jsonl
# ------------------------------------------------------------
possible_jsonl_paths = [
    "../2_Clinical_Verification/kaggle_partA/partA_final.jsonl",
    "./Custom_Tropical_Project/2_Clinical_Verification/kaggle_partA/partA_final.jsonl",
    "/content/kaggle_partA/partA_final.jsonl",
    "/content/partA_final.jsonl",
    "/content/drive/MyDrive/partA_final.jsonl",
    "/content/drive/MyDrive/kaggle_partA/partA_final.jsonl",
]
CASES_FILE = next((p for p in possible_jsonl_paths if os.path.exists(p)), None)
if CASES_FILE is None:
    raise FileNotFoundError("❌ partA_final.jsonl not found! Make sure partA_final.jsonl or archive.zip is present.")

all_cases = []
with open(CASES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        all_cases.append(json.loads(line))

# ------------------------------------------------------------
# C. Locate extracted_triplets folder & load into RAM cache
# ------------------------------------------------------------
possible_triplets_dirs = [
    "./kg_extraction/extracted_triplets",
    "../3_LightRAG_Knowledge_Graph/kg_extraction/extracted_triplets",
    "./extracted_triplets",
    "/content/extracted_triplets",
    "/content/drive/MyDrive/extracted_triplets",
    "/content/drive/MyDrive/kg_extraction/extracted_triplets",
]
TRIPLETS_DIR = next((p for p in possible_triplets_dirs if os.path.exists(p)), None)

EXTRACTION_CACHE = {}
if TRIPLETS_DIR:
    triplet_files = glob.glob(os.path.join(TRIPLETS_DIR, "*.json"))
    for path in triplet_files:
        cid = os.path.basename(path).replace(".json", "")
        try:
            with open(path, "r", encoding="utf-8") as f:
                EXTRACTION_CACHE[cid] = json.load(f)
        except Exception:
            pass

print(f"✅ Found dataset: {CASES_FILE} ({len(all_cases)} cases)")
print(f"✅ Found triplets directory: {TRIPLETS_DIR}")
print(f"   Loaded {len(EXTRACTION_CACHE)} pre-extracted cases into memory cache")
if len(EXTRACTION_CACHE) == 0:
    print("⚠️ WARNING: No extracted triplets found in cache! Ensure extracted_triplets/ has .json files or extracted_triplets.zip is provided.")

---
## Step 6: Build the Knowledge Graph (~10–15 minutes) 🏗️

Ingestion runs on Colab's **fast local NVMe SSD** (~0.3s–0.5s per case).
After all cases are processed, the finished KG is automatically copied to Google Drive once.

- **1 Case = 1 Chunk (`chunk_token_size=4096`):** Preserves complete clinical diagnostic context per patient.
- **Mock LLM Extraction (0.001s):** Instantly retrieves frontier-extracted triplets from memory cache.
- **Zero LLM Bottleneck:** Fast-paths gleaning & merge summarization without calling local LLM.
- **bge-m3 Embedding (GPU):** Embeds entities, relations, and full patient narratives into 1024-dim vector space.
- **Auto-Resume:** Automatically detects and skips any cases already indexed in the graph.

In [ ]:
import asyncio, time, os, shutil, re, json
from functools import partial
from tqdm.auto import tqdm
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_embed
from lightrag.utils import EmbeddingFunc

try:
    import nest_asyncio
except ImportError:
    import nest_asyncio2 as nest_asyncio
nest_asyncio.apply()

# ========== CONFIGURATION ==========
WORKSPACE = "tropical_kg"          # Fixed workspace name (preserves graph)
EMBEDDING_MODEL = "bge-m3"
EMBED_DIM = 1024
QUERY_LLM_MODEL = "qwen2.5:3b"      # Used strictly for Step 7 diagnostic queries
CONCURRENCY = 8
# ===================================

os.makedirs(LOCAL_STORAGE, exist_ok=True)
print(f"📁 Working directory: {LOCAL_STORAGE}")

# --- Mock LLM Bridge (Complete 4-Stage State Machine) ---
async def mock_llm_complete(prompt, system_prompt=None, history_messages=[], **kwargs):
    """
    Dual-mode LLM Function:
    - Ingestion: Returns pre-extracted triplets in 0.001s; handles gleaning & merges instantly.
    - Querying (Step 7): Delegates to Ollama for real-time doctor response synthesis.
    """
    # 1. Ingestion: Gleaning prompt ("did you miss anything?") -> Instant empty return
    if "missed or incorrectly described" in prompt:
        return '{"entities": [], "relationships": []}'

    # 2. Ingestion: Entity & Relationship extraction prompt
    if "Extract entities and relationships" in prompt or "---Input Text---" in prompt:
        match = re.search(r"Case ID:\s*(\S+)", prompt)
        if match:
            case_id = match.group(1).rstrip(" -")
            if case_id in EXTRACTION_CACHE:
                return json.dumps(EXTRACTION_CACHE[case_id])
        # Multi-chunk continuations or already-extracted cases return empty JSON
        return '{"entities": [], "relationships": []}'

    # 3. Ingestion: Multi-Case Entity/Relationship Description Summarization
    if "synthesize a list of descriptions" in prompt or "Description List:" in prompt:
        descriptions = re.findall(r'\{\"Description\":\s*\"(.*?)\"\}', prompt)
        if descriptions:
            seen = set()
            unique_descs = []
            for d in descriptions:
                clean_d = d.strip()
                if clean_d and clean_d not in seen:
                    seen.add(clean_d)
                    unique_descs.append(clean_d)
            return " | ".join(unique_descs[:15])
        return "Clinical findings and disease presentation across tropical patient cases."

    # 4. Query time (Step 7): Delegate to Ollama for diagnostic answer synthesis
    try:
        from lightrag.llm.ollama import ollama_model_complete
        return await ollama_model_complete(
            prompt, system_prompt=system_prompt, history_messages=history_messages, **kwargs
        )
    except Exception as e:
        if "high_level_keywords" in prompt:
            return '{"high_level_keywords": [], "low_level_keywords": []}'
        return f"Retrieved Knowledge Graph context, but query LLM was unavailable: {str(e)}"

# --- Initialize LightRAG on local SSD with Meaningful Architecture Settings ---
print(f"Using workspace: {WORKSPACE}")
print(f"Using Mock LLM for Ingestion (Instant) + {EMBEDDING_MODEL} Embedding ({EMBED_DIM} dims)")

rag = LightRAG(
    working_dir=LOCAL_STORAGE,
    workspace=WORKSPACE,
    llm_model_func=mock_llm_complete,
    llm_model_name=QUERY_LLM_MODEL,
    llm_model_max_async=CONCURRENCY,
    embedding_func_max_async=4,
    default_embedding_timeout=120,
    entity_extraction_use_json=True,
    enable_llm_cache=False,
    chunk_token_size=4096,                          # 1 Case = 1 Chunk: Keeps whole patient narrative together
    chunk_overlap_token_size=100,
    entity_extract_max_gleaning=0,                  # Skip redundant gleaning (frontier triplets already complete)
    force_llm_summary_on_merge=999999,              # Prevent premature LLM merge rewrite
    summary_max_tokens=8192,                        # Accumulate multi-case clinical evidence directly
    summary_context_size=16384,
    enable_llm_cache_for_entity_extract=False,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=4096,
        func=partial(ollama_embed.func, embed_model=EMBEDDING_MODEL)
    )
)

print("Initializing LightRAG storages...")
await rag.initialize_storages()
print("✅ Storages initialized\n")

# --- Inspect already indexed cases (Auto-Resume) ---
indexed_doc_file = os.path.join(LOCAL_STORAGE, WORKSPACE, "kv_store_full_docs.json")
already_indexed = set()
if os.path.exists(indexed_doc_file):
    try:
        with open(indexed_doc_file, "r", encoding="utf-8") as f:
            already_indexed = set(json.load(f).keys())
    except Exception:
        pass

cases_to_process = [c for c in all_cases if c["case_id"] not in already_indexed]
print(f"📊 Cases already in Knowledge Graph: {len(already_indexed)} / {len(all_cases)}")
print(f"🎯 Cases remaining to process: {len(cases_to_process)}\n")

# --- Ingest cases ---
if len(cases_to_process) > 0:
    start_total = time.time()
    for row in tqdm(cases_to_process, desc="Embedding & Building KG", unit="case"):
        case_id = row["case_id"]
        title = row.get("title", "")
        present_diseases = ", ".join(row.get("present", [])) or "Unknown"
        main_problem = row.get("main_problem", "N/A")

        evidence_quotes = []
        for d, info in row.get("candidates", {}).items():
            if info.get("status") == "present" and info.get("evidence"):
                evidence_quotes.extend(info["evidence"])
        evidence_str = " ".join(evidence_quotes) if evidence_quotes else "N/A"

        img_findings = [
            f"[{img.get('image_type', 'image')}/{img.get('image_subtype', 'finding')}]: {img.get('caption', '')}"
            for img in row.get("images", []) if img.get("caption")
        ]
        img_str = " | ".join(img_findings) if img_findings else "No imaging panels."

        case_document = (
            f"Case ID: {case_id} - {title}\n"
            f"Confirmed Diagnosis: {present_diseases}\n"
            f"Primary Clinical Problem: {main_problem}\n"
            f"Diagnostic Evidence: {evidence_str}\n"
            f"Imaging Findings: {img_str}\n\n"
            f"Clinical Narrative:\n{row.get('case_text', '')}"
        )

        await rag.ainsert(case_document, ids=case_id, file_paths=f"{case_id}.txt")

    total_time = time.time() - start_total
    print(f"\n{'='*60}")
    print(f"✅ Knowledge Graph built! Ingested {len(cases_to_process)} cases in {total_time/60:.1f} minutes")
    print(f"   Avg: {total_time/len(cases_to_process):.2f}s per case")
    print(f"{'='*60}")
else:
    print("🎉 All cases are already ingested in the Knowledge Graph!")

# --- Copy finished KG to Google Drive ---
if IN_COLAB and DRIVE_STORAGE != LOCAL_STORAGE:
    print(f"\n💾 Copying Knowledge Graph to Google Drive...")
    os.makedirs(DRIVE_STORAGE, exist_ok=True)
    os.system(f"cp -r {LOCAL_STORAGE}/* {DRIVE_STORAGE}/")
    print(f"✅ Saved to: {DRIVE_STORAGE}")

---
## Step 7: Query the Knowledge Graph 🔍

In [ ]:
test_query = "What are the common symptoms, lab abnormalities, and risk factors for Dengue fever and Tuberculosis in tropical patients?"
print(f"Query: {test_query}\n")

response = await rag.aquery(test_query, param=QueryParam(mode="hybrid"))

print("=" * 60)
print("LightRAG Diagnostic Response (Hybrid Mode)")
print("=" * 60)
print(response)

### Try More Queries
Change the query below and re-run to explore the Knowledge Graph.

In [ ]:
# Try different queries!
my_query = "What imaging findings distinguish pulmonary tuberculosis from bacterial pneumonia?"

response = await rag.aquery(my_query, param=QueryParam(mode="hybrid"))
print(response)

---
## Step 8: Download Knowledge Graph to Your Mac
Downloads `tropical_kg_storage.zip` to your computer.
Unzip it into your local `Datasets/tropical_kg_storage/` folder to use offline!

In [ ]:
import os, shutil, sys

zip_path = "./tropical_kg_storage"
shutil.make_archive(zip_path, 'zip', LOCAL_STORAGE)
print(f"✅ Knowledge Graph archived to: {zip_path}.zip")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("Downloading to browser...")